In [ ]:
import torch
import torchvision
import torch.nn as nn
import os, numpy as np, matplotlib.pyplot as plt, pandas as pd, cv2, random
from PIL import Image
from model import BrainTumorModel
from utils import CLAHETransform

In [3]:
train_transform = torchvision.transforms.Compose([
    # RandomResizedCrop helps with scale/position variations in MR slices
    torchvision.transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),
    torchvision.transforms.RandomRotation(degrees=15),
    torchvision.transforms.RandomHorizontalFlip(p=0.5),
    torchvision.transforms.ColorJitter(brightness=0.08, contrast=0.08),
    torchvision.transforms.RandomApply([CLAHETransform()], p=0.5),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Keep the test transform deterministic
testing_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
testing_dataset = torchvision.datasets.ImageFolder(root=os.path.join('Testing'), transform=testing_transform)
training_dataset = torchvision.datasets.ImageFolder(root=os.path.join('Training'), transform=train_transform)

testing_loader = torch.utils.data.DataLoader(testing_dataset, batch_size=32, shuffle=True)
training_loader = torch.utils.data.DataLoader(training_dataset, batch_size=32, shuffle=True)

In [ ]:
import math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BrainTumorModel(num_classes=4)
model.to(device)
criterion = nn.CrossEntropyLoss()
base_lr = 5e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=base_lr, weight_decay=0.05)
scaler = torch.amp.GradScaler()

num_epochs = 30
log_interval = 10
warmup_ratio = 0.05  # proportion of total steps used for linear warmup

# compute total steps (batches) to build the schedule
total_steps = num_epochs * len(training_loader)
warmup_steps = max(1, int(total_steps * warmup_ratio))

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)
cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=max(1, total_steps - warmup_steps),
    eta_min=5e-7
)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[warmup_scheduler, cosine_scheduler],
    milestones=[warmup_steps]
)

history = {
    'epoch': [],
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': []
}

global_step = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_idx, (images, labels) in enumerate(training_loader, 1):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        with torch.amp.autocast(device_type='cuda' if device.type == 'cuda' else 'cpu'):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # step scheduler per batch (after optimizer.step)
        scheduler.step()
        global_step += 1

        train_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

        if batch_idx % log_interval == 0 or batch_idx == len(training_loader):
            batch_loss = train_loss / train_total
            batch_acc = train_correct / train_total
            current_lr = optimizer.param_groups[0]['lr']
            print(f"Epoch {epoch + 1}/{num_epochs} | Batch {batch_idx}/{len(training_loader)} | "
                  f"Train Loss: {batch_loss:.4f} | Train Acc: {batch_acc:.4f} | LR: {current_lr:.6e}")

    epoch_loss = train_loss / train_total
    epoch_acc = train_correct / train_total

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in testing_loader:
            images = images.to(device)
            labels = labels.to(device)
            with torch.amp.autocast(device_type='cuda' if device.type == 'cuda' else 'cpu'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss / val_total
    val_acc = val_correct / val_total

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(epoch_loss)
    history['train_acc'].append(epoch_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(f"Epoch {epoch + 1}/{num_epochs} completed | "
          f"Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

history_df = pd.DataFrame(history)
print('\nTraining history:')
print(history_df)


In [7]:
torch.save(model.state_dict(), 'brain_tumor_model.pth')